<a href="https://colab.research.google.com/github/melatisurya/Data_Metopen_Klasifikasi_Penyakit_Daun_Jagung/blob/main/Maize_ResNet50_Revisi_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasifikasi Penyakit Daun Jagung dengan ResNet50 (versi perbaikan)

Notebook ini menjalankan pipeline yang sudah diperbaiki sesuai catatan reviewer:

1. **Praproses benar**: `resnet50.preprocess_input` (bukan pembagian 255).
2. **Validasi silang sah**: model diinisialisasi ulang pada tiap fold; augmentasi hanya dibuat dari citra latih fold tersebut.
3. **Fine-tuning** blok terakhir (`conv5_block3`) setelah kepala klasifikasi dilatih.
4. Data uji (64 citra) tidak dipakai untuk memilih model; model terbaik dipilih dari akurasi validasi.
5. Metrik: akurasi (benar/n), presisi/recall/F1 berbobot, AUC OvR makro, interval Wilson 95%.

**Cara pakai:** Runtime > Change runtime type (GPU opsional, CPU juga bisa) lalu jalankan sel dari atas ke bawah. Unggah zip dataset (folder berisi `Hawar`, `Karat`, `Sehat`) saat diminta. Hasil (CSV, gambar, ringkasan) ada di folder `hasil_resnet50` dan diunduh di sel terakhir.

> Catatan efisiensi: lapisan sebelum `conv5_block3` selalu beku, sehingga keluarannya di-cache sekali. Melatih `conv5_block3` + kepala di atas cache ini setara matematis dengan melatih model utuh dengan lapisan awal beku (BatchNorm dibekukan), tetapi jauh lebih cepat.

> Bagian 11-13 (kontrol akuisisi, perbandingan arsitektur ringan, dan dataset publik PlantVillage) menjawab catatan reviewer 2 dan 3. Bagian 12-13 membutuhkan unduhan bobot ImageNet tambahan dan dataset `plant_village`.

## 1. Instalasi

In [1]:
!pip -q install split-folders opencv-python-headless

## 2. Unggah dan ekstrak dataset

Pilih berkas zip dataset (misalnya `archive (2).zip`). Jika nama folder di dalam zip berbeda, ubah `DATA_DIR` di sel berikutnya.

In [2]:
from google.colab import files
import zipfile, os
up = files.upload()   # pilih zip dataset
for name in up:
    zipfile.ZipFile(name).extractall('data')
print(os.listdir('data'))

Saving archive.zip to archive.zip
['Penyakit Daun Jagung New']


## 3. Pengaturan

`K_AUG` = jumlah salinan augmentasi per citra latih, `N_FOLDS` = jumlah fold, `EPOCHS_FT` = epoch maksimum fine-tuning.

In [3]:
import os
os.environ['DATA_DIR'] = 'data/Penyakit Daun Jagung New'   # sesuaikan jika perlu
os.environ['OUT_DIR']  = 'hasil_resnet50'
os.environ['WEIGHTS']  = 'imagenet'                        # bobot ImageNet resmi Keras
os.environ['K_AUG']    = '3'
os.environ['N_FOLDS']  = '5'
os.environ['EPOCHS_FT'] = '8'

## 4. Impor dan konstanta

In [4]:
import os, time, json, math
import numpy as np, pandas as pd, cv2
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression
import splitfolders
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_DIR = os.environ.get("DATA_DIR", "data/Penyakit Daun Jagung New")
OUT_DIR  = os.environ.get("OUT_DIR", "hasil_resnet50")
WEIGHTS  = os.environ.get("WEIGHTS", "imagenet")
K_AUG    = int(os.environ.get("K_AUG", 3))         # jumlah salinan augmentasi per citra latih
N_FOLDS  = int(os.environ.get("N_FOLDS", 5))
SEED     = 69
IMG      = 224
EPOCHS_HEAD, PAT_HEAD, LR_HEAD = 75, 10, 1e-4
EPOCHS_FT,   PAT_FT,   LR_FT   = int(os.environ.get("EPOCHS_FT", 8)), 3, 5e-5
BATCH    = 16
os.makedirs(OUT_DIR, exist_ok=True)
T0 = time.time()
def log(*a): print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
keras.utils.set_random_seed(SEED)


## 5. Data dan pembagian 80:20 (seed 69)

In [5]:
# ------------------------------------------------------------------ 1. data dan pembagian 80:20 (seed 69)
split_dir = os.path.join(OUT_DIR, "split")
if not os.path.exists(split_dir):
    splitfolders.ratio(DATA_DIR, output=split_dir, seed=SEED, ratio=(.8, .2), move=False)
classes = sorted(d for d in os.listdir(os.path.join(split_dir, "train")))
CN = {"Hawar": "Leaf blight", "Karat": "Leaf rust", "Sehat": "Healthy"}
def load_split(name):
    X, y = [], []
    for i, c in enumerate(classes):
        for f in sorted(os.listdir(os.path.join(split_dir, name, c))):
            im = cv2.imread(os.path.join(split_dir, name, c, f))
            im = cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)   # area interpolation
            X.append(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); y.append(i)
    return np.stack(X), np.array(y)
Xtr, ytr = load_split("train"); Xte, yte = load_split("val")
log("kelas", classes, "| latih", Xtr.shape[0], np.bincount(ytr), "| uji", Xte.shape[0], np.bincount(yte))


Copying files: 310 files [00:00, 605.26 files/s]


[    39s] kelas ['Hawar', 'Karat', 'Sehat'] | latih 246 [85 81 80] | uji 64 [22 21 21]


## 6. Augmentasi (parameter mengikuti Augmentor)

In [6]:
# ------------------------------------------------------------------ 2. augmentasi (parameter mengikuti Augmentor)
def augment(img, rng):
    h, w = img.shape[:2]
    # skew 0,15
    m = 0.15 * rng.uniform(0.5, 1.0)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    d = rng.integers(0, 8); s = m * w
    dst = src.copy()
    if d == 0: dst[0, 0] += s; dst[3, 0] += s            # miring ke kanan (sisi kiri)
    elif d == 1: dst[1, 0] -= s; dst[2, 0] -= s
    elif d == 2: dst[0, 1] += s; dst[1, 1] += s
    elif d == 3: dst[2, 1] -= s; dst[3, 1] -= s
    elif d == 4: dst[0] += [s, s]
    elif d == 5: dst[1] += [-s, s]
    elif d == 6: dst[2] += [-s, -s]
    else: dst[3] += [s, -s]
    out = cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h), borderMode=cv2.BORDER_REFLECT)
    # kecerahan 0,75-1,25
    out = np.clip(out.astype(np.float32) * rng.uniform(0.75, 1.25), 0, 255).astype(np.uint8)
    # rotasi +-3 derajat dan zoom 0,95-1,05
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-3, 3), rng.uniform(0.95, 1.05))
    out = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if rng.random() < 0.5: out = out[:, ::-1]
    return np.ascontiguousarray(out)
rng = np.random.default_rng(SEED)
Xaug = np.stack([augment(Xtr[i], rng) for i in range(len(Xtr)) for _ in range(K_AUG)])
src_idx = np.repeat(np.arange(len(Xtr)), K_AUG)               # sumber tiap salinan augmentasi
log("salinan augmentasi:", Xaug.shape[0])


[    41s] salinan augmentasi: 738


## 7. ResNet50 dan cache fitur

Bagian ini paling lama (ekstraksi fitur citra asli, salinan augmentasi, dan uji praproses lama `/255`).

In [7]:
# ------------------------------------------------------------------ 3. ResNet50 dan cache fitur
base = keras.applications.ResNet50(include_top=False, weights=WEIGHTS, input_shape=(IMG, IMG, 3))
f2 = keras.Model(base.input, base.get_layer("conv5_block2_out").output)
b3 = keras.Model(base.get_layer("conv5_block2_out").output, base.output)    # conv5_block3 saja
b3_init = b3.get_weights()
def feats(X, mode):
    X = X.astype("float32")
    X = keras.applications.resnet50.preprocess_input(X) if mode == "correct" else X / 255.0
    return f2.predict(X, batch_size=16, verbose=0)
cache = os.path.join(OUT_DIR, "cache.npz")
if os.path.exists(cache):
    z = np.load(cache); F2tr, F2aug, F2te, Pbug_tr, Pbug_aug, Pbug_te = [z[k] for k in
        ("F2tr", "F2aug", "F2te", "Pbug_tr", "Pbug_aug", "Pbug_te")]
else:
    F2tr = feats(Xtr, "correct").astype("float16"); log("fitur latih selesai")
    F2aug = feats(Xaug, "correct").astype("float16"); log("fitur augmentasi selesai")
    F2te = feats(Xte, "correct").astype("float16"); log("fitur uji selesai")
    pool = lambda F2: b3.predict(F2, batch_size=32, verbose=0).mean((1, 2))
    Pbug_tr = pool(feats(Xtr, "bug")); Pbug_aug = pool(feats(Xaug, "bug")); Pbug_te = pool(feats(Xte, "bug"))
    log("fitur /255 (uji kesalahan praproses lama) selesai")
    np.savez(cache, F2tr=F2tr, F2aug=F2aug, F2te=F2te, Pbug_tr=Pbug_tr, Pbug_aug=Pbug_aug, Pbug_te=Pbug_te)
pool = lambda F2: b3.predict(F2.astype("float32"), batch_size=32, verbose=0).mean((1, 2))
Ptr, Paug, Pte = pool(F2tr), pool(F2aug), pool(F2te)           # fitur beku (GAP conv5_block3_out)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[    84s] fitur latih selesai
[   194s] fitur augmentasi selesai
[   203s] fitur uji selesai
[   405s] fitur /255 (uji kesalahan praproses lama) selesai


## 8. Validasi silang 5-fold: kepala (F) lalu fine-tuning conv5_block3 (T)

Konfigurasi: `F-aug-/255` (praproses lama), `F-noaug`, `F-aug`, `T-noaug`, `T-aug`.

In [17]:
# ------------------------------------------------------------------ 4. model kepala dan fine-tuning
def head():
    return keras.Sequential([keras.layers.Input((2048,)), keras.layers.Dropout(0.5),
                             keras.layers.Dense(512, activation="relu"), keras.layers.Dropout(0.5),
                             keras.layers.Dense(3, activation="softmax")])
def pred_p(m, X, bs=64):
    """Prediksi langsung (tanpa membuat tf.function baru tiap model) agar tidak muncul peringatan retracing."""
    X = np.asarray(X, dtype="float32")
    return np.concatenate([m(X[i:i + bs], training=False).numpy() for i in range(0, len(X), bs)])
def stop(p): return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=p, restore_best_weights=True)]
def metrics(y, proba):
    pred = proba.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)
    return dict(acc=accuracy_score(y, pred), prec_w=p, rec_w=r, f1_w=f,
                auc=roc_auc_score(y, proba, multi_class="ovr", average="macro"))
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)

rows, store = [], {}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (itr, iva) in enumerate(skf.split(Xtr, ytr), 1):
    for aug in (False, True):
        keras.utils.set_random_seed(SEED + fold)
        # data latih fold: citra asli fold + salinan augmentasi HANYA dari citra asli fold ini
        if aug:
            sel = np.isin(src_idx, itr)
            Xp = np.concatenate([Ptr[itr], Paug[sel]]); Xb = np.concatenate([Pbug_tr[itr], Pbug_aug[sel]])
            Xf = np.concatenate([F2tr[itr], F2aug[sel]]); yy = np.concatenate([ytr[itr], ytr[src_idx[sel]]])
        else:
            Xp, Xb, Xf, yy = Ptr[itr], Pbug_tr[itr], F2tr[itr], ytr[itr]
        tag = "aug" if aug else "noaug"
        # ---- F: kepala saja (backbone beku), kepala diinisialisasi ulang tiap fold
        h = head(); h.compile(keras.optimizers.Adam(LR_HEAD), "sparse_categorical_crossentropy", metrics=["accuracy"])
        hist = h.fit(Xp, yy, validation_data=(Ptr[iva], ytr[iva]), epochs=EPOCHS_HEAD, batch_size=BATCH,
                     callbacks=stop(PAT_HEAD), verbose=0)
        pv, pt = pred_p(h, Ptr[iva]), pred_p(h, Pte)
        cfg = f"F-{tag}"
        rows.append(dict(config=cfg, fold=fold, val_acc=accuracy_score(ytr[iva], pv.argmax(1)),
                         val_loss=min(hist.history["val_loss"]), epochs=len(hist.history["loss"]),
                         **{"test_" + k: v for k, v in metrics(yte, pt).items()}))
        store[(cfg, fold)] = dict(proba=pt, hist=hist.history)
        hw = h.get_weights()
        # ---- uji kesalahan praproses lama (/255) pada desain F lama: hanya skenario augmentasi
        if aug:
            hb = head(); hb.compile(keras.optimizers.Adam(LR_HEAD), "sparse_categorical_crossentropy", metrics=["accuracy"])
            hh = hb.fit(Xb, yy, validation_data=(Pbug_tr[iva], ytr[iva]), epochs=EPOCHS_HEAD, batch_size=BATCH,
                        callbacks=stop(PAT_HEAD), verbose=0)
            pv2, pt2 = pred_p(hb, Pbug_tr[iva]), pred_p(hb, Pbug_te)
            rows.append(dict(config="F-aug-/255", fold=fold, val_acc=accuracy_score(ytr[iva], pv2.argmax(1)),
                             val_loss=min(hh.history["val_loss"]), epochs=len(hh.history["loss"]),
                             **{"test_" + k: v for k, v in metrics(yte, pt2).items()}))
        # ---- T: fine-tuning conv5_block3 (BatchNorm beku) + kepala, mulai dari kepala F
        inp = keras.Input((7, 7, 2048))
        blk = keras.models.clone_model(b3); blk.set_weights(b3_init)
        for l in blk.layers:
            if isinstance(l, keras.layers.BatchNormalization): l.trainable = False
        hd = head(); hd.set_weights(hw)
        out = hd(keras.layers.GlobalAveragePooling2D()(blk(inp)))
        ft = keras.Model(inp, out)
        ft.compile(keras.optimizers.Adam(LR_FT), "sparse_categorical_crossentropy", metrics=["accuracy"])
        hf = ft.fit(Xf.astype("float32"), yy, validation_data=(F2tr[iva].astype("float32"), ytr[iva]),
                    epochs=EPOCHS_FT, batch_size=BATCH, callbacks=stop(PAT_FT), verbose=0)
        pv3 = pred_p(ft, F2tr[iva].astype("float32")); pt3 = pred_p(ft, F2te.astype("float32"))
        cfg = f"T-{tag}"
        rows.append(dict(config=cfg, fold=fold, val_acc=accuracy_score(ytr[iva], pv3.argmax(1)),
                         val_loss=min(hf.history["val_loss"]), epochs=len(hf.history["loss"]),
                         **{"test_" + k: v for k, v in metrics(yte, pt3).items()}))
        store[(cfg, fold)] = dict(proba=pt3, hist=hf.history)
        log(f"fold {fold} {tag}:", {r["config"]: round(r["test_acc"], 3) for r in rows[-(3 if aug else 2):]})
        keras.backend.clear_session()
        pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "hasil_per_fold.csv"), index=False)

[  3926s] fold 1 noaug: {'F-noaug': 0.938, 'T-noaug': 0.906}
[  4161s] fold 1 aug: {'F-aug': 0.922, 'F-aug-/255': 0.484, 'T-aug': 0.938}
[  4252s] fold 2 noaug: {'F-noaug': 0.922, 'T-noaug': 0.953}
[  4567s] fold 2 aug: {'F-aug': 0.938, 'F-aug-/255': 0.422, 'T-aug': 0.953}
[  4633s] fold 3 noaug: {'F-noaug': 0.922, 'T-noaug': 0.922}
[  4925s] fold 3 aug: {'F-aug': 0.953, 'F-aug-/255': 0.438, 'T-aug': 0.938}
[  5001s] fold 4 noaug: {'F-noaug': 0.938, 'T-noaug': 0.938}
[  5159s] fold 4 aug: {'F-aug': 0.938, 'F-aug-/255': 0.5, 'T-aug': 0.969}
[  5215s] fold 5 noaug: {'F-noaug': 0.953, 'T-noaug': 0.953}
[  5521s] fold 5 aug: {'F-aug': 0.938, 'F-aug-/255': 0.359, 'T-aug': 0.938}


## 9. Ringkasan, model terbaik (dipilih dari validasi), tabel, dan gambar

In [9]:
# ------------------------------------------------------------------ 5. ringkasan
R = pd.DataFrame(rows)
order = ["F-aug-/255", "F-noaug", "F-aug", "T-noaug", "T-aug"]
def ms(s, pct=True):
    return f"{100*s.mean():.1f} ± {100*s.std(ddof=1):.1f}" if pct else f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"
summ = pd.DataFrame([{"Konfigurasi": c, "Val acc (%)": ms(g.val_acc), "Test acc (%)": ms(g.test_acc),
                      "Test precision (w)": ms(g.test_prec_w, False), "Test F1 (w)": ms(g.test_f1_w, False),
                      "Test AUC": ms(g.test_auc, False), "Epoch": f"{g.epochs.mean():.1f}"}
                     for c in order for g in [R[R.config == c]]])
summ.to_csv(os.path.join(OUT_DIR, "ringkasan.csv"), index=False)
# model terbaik DIPILIH dari rerata akurasi validasi (bukan uji), fold terbaiknya dari akurasi validasi
cfgs = [c for c in order if c != "F-aug-/255"]
best_cfg = R[R.config.isin(cfgs)].groupby("config").val_acc.mean().idxmax()
bf = int(R[R.config == best_cfg].sort_values("val_acc").iloc[-1].fold)
proba = store[(best_cfg, bf)]["proba"]; pred = proba.argmax(1)
n_ok = int((pred == yte).sum()); lo, hi = wilson(n_ok, len(yte))
cm = confusion_matrix(yte, pred)
pc = precision_recall_fscore_support(yte, pred, zero_division=0)
pd.DataFrame({"Kelas": [CN.get(c, c) for c in classes], "Precision": pc[0], "Recall": pc[1], "F1": pc[2], "Support": pc[3]}
             ).to_csv(os.path.join(OUT_DIR, "per_kelas_terbaik.csv"), index=False)
# matriks kebingungan gabungan seluruh fold untuk konfigurasi terbaik
cm_all = sum(confusion_matrix(yte, store[(best_cfg, f)]["proba"].argmax(1)) for f in range(1, N_FOLDS + 1))
json.dump(dict(best_cfg=best_cfg, best_fold=bf, test_acc=n_ok / len(yte), n_correct=n_ok, wilson=[lo, hi],
               cm=cm.tolist(), cm_pooled=cm_all.tolist(), classes=classes), open(os.path.join(OUT_DIR, "terbaik.json"), "w"))
# baseline isyarat tingkat rendah: regresi logistik pada rerata RGB (uji kebocoran/pengganggu akuisisi)
mt = lambda X: X.reshape(len(X), -1, 3).mean(1) / 255.0
lr = LogisticRegression(max_iter=1000).fit(mt(Xtr), ytr)
json.dump(dict(rgb_only_test_acc=float(lr.score(mt(Xte), yte))), open(os.path.join(OUT_DIR, "baseline_rgb.json"), "w"))
# gambar
fig, ax = plt.subplots(figsize=(7, 3.4))
for c, mk in zip(cfgs + ["F-aug-/255"], "osd^v"):
    g = R[R.config == c].sort_values("fold")
    ax.plot(g.fold, 100 * g.test_acc, marker=mk, label=c)
ax.set_xlabel("Fold"); ax.set_ylabel("Test accuracy (%)"); ax.set_xticks(range(1, N_FOLDS + 1)); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_acc_per_fold.png"), dpi=200)
fig, ax = plt.subplots(figsize=(4.2, 3.8)); ax.imshow(cm, cmap="Blues")
lab = [CN.get(c, c) for c in classes]; ax.set_xticks(range(3)); ax.set_xticklabels(lab, fontsize=8); ax.set_yticks(range(3)); ax.set_yticklabels(lab, fontsize=8)
for i in range(3):
    for j in range(3): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_confusion.png"), dpi=200)
hs = store[(best_cfg, bf)]["hist"]
fig, ax = plt.subplots(1, 2, figsize=(7, 3)); ax[0].plot(hs["loss"], label="train"); ax[0].plot(hs["val_loss"], label="val"); ax[0].set_title("Loss", fontsize=9)
ax[1].plot(hs["accuracy"], label="train"); ax[1].plot(hs["val_accuracy"], label="val"); ax[1].set_title("Accuracy", fontsize=9)
for a in ax: a.set_xlabel("Epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_curves.png"), dpi=200)
log("SELESAI | terbaik:", best_cfg, "fold", bf, "| uji", n_ok, "/", len(yte), f"| Wilson {lo:.1f}-{hi:.1f}")
print(summ.to_string(index=False))


[  2158s] SELESAI | terbaik: T-aug fold 5 | uji 60 / 64 | Wilson 85.0-97.5
Konfigurasi Val acc (%) Test acc (%) Test precision (w)   Test F1 (w)      Test AUC Epoch
 F-aug-/255  49.6 ± 8.4   44.1 ± 5.6      0.462 ± 0.146 0.364 ± 0.100 0.717 ± 0.027  23.2
    F-noaug  94.3 ± 1.7   93.4 ± 1.3      0.935 ± 0.013 0.934 ± 0.013 0.983 ± 0.004  64.2
      F-aug  94.3 ± 3.0   93.8 ± 1.1      0.939 ± 0.011 0.937 ± 0.011 0.987 ± 0.006  42.2
    T-noaug  94.7 ± 1.8   93.4 ± 2.0      0.936 ± 0.020 0.934 ± 0.020 0.984 ± 0.002   5.2
      T-aug  95.1 ± 2.3   94.7 ± 1.4      0.950 ± 0.015 0.947 ± 0.014 0.992 ± 0.005   6.2


## 11. Kontrol akuisisi: kelas sehat diambil pada waktu berbeda (catatan reviewer 2)

Pengambilan ulang data sehat pada kondisi yang sama tetap solusi terbaik dan harus dilakukan di lapangan. Sambil menunggu, sel-sel berikut menguji seberapa besar model bergantung pada isyarat waktu/warna:

* **11a** tabel tanggal dan jam pemotretan per kelas dari Exif (dasar untuk membagi per sesi).
* **11b** uji *session hold-out*: citra sehat dari sesi tanggal minoritas dikeluarkan dari seluruh pelatihan dan hanya dipakai untuk uji. Jika model benar-benar mengenali daun sehat, citra itu tetap diprediksi sehat.
* **11c** kontrol warna: ekstraksi fitur diulang dengan citra grayscale dan dengan standardisasi per citra (menghilangkan perbedaan kecerahan/warna global). Jika akurasi tetap mirip, ketergantungan pada isyarat warna global lebih kecil.

Catatan: kontrol ini tidak menggantikan pengambilan data ulang. Kelas penyakit seluruhnya diambil Maret, sehingga tanggal tidak bisa dipisahkan penuh dari kelas.

In [10]:
# ---- 11a. metadata Exif per citra (urutan sama dengan load_split: latih lalu uji, kelas terurut, berkas terurut)
from PIL import Image
from sklearn.model_selection import train_test_split

def exif_dt(path):
    try:
        ex = Image.open(path).getexif()
        v = ex.get_ifd(0x8769).get(36867) or ex.get(306)      # DateTimeOriginal atau DateTime
        return v if v else "NA"
    except Exception:
        return "NA"

meta = []
for split in ("train", "val"):
    for c in classes:
        for f in sorted(os.listdir(os.path.join(split_dir, split, c))):
            dt = exif_dt(os.path.join(split_dir, split, c, f))
            meta.append(dict(split=split, kelas=c, berkas=f, tanggal=dt[:10].replace(":", "-"), jam=dt[11:16]))
meta = pd.DataFrame(meta)
assert len(meta) == len(Xtr) + len(Xte), "urutan metadata tidak cocok dengan data"
meta.to_csv(os.path.join(OUT_DIR, "meta_exif.csv"), index=False)
print(pd.crosstab(meta.kelas, meta.tanggal))
print(meta.groupby("kelas").jam.agg(["min", "max"]))

def head_d(dim):
    return keras.Sequential([keras.layers.Input((dim,)), keras.layers.Dropout(0.5),
                             keras.layers.Dense(512, activation="relu"), keras.layers.Dropout(0.5),
                             keras.layers.Dense(3, activation="softmax")])
def fit_head(X, y, Xv=None, yv=None, seed=SEED):
    """Latih kepala klasifikasi; jika Xv tidak diberikan, 20% X (stratifikasi) dipakai sebagai validasi early stopping."""
    if Xv is None:
        X, Xv, y, yv = train_test_split(X, y, test_size=.2, stratify=y, random_state=seed)
    keras.utils.set_random_seed(seed)
    h = head_d(X.shape[1])
    h.compile(keras.optimizers.Adam(LR_HEAD), "sparse_categorical_crossentropy", metrics=["accuracy"])
    h.fit(X, y, validation_data=(Xv, yv), epochs=EPOCHS_HEAD, batch_size=BATCH, callbacks=stop(PAT_HEAD), verbose=0)
    return h


tanggal  2023-03-14  2023-03-15  2023-03-16
kelas                                      
Hawar             0          49          58
Karat             7          95           0
Sehat             0         101           0
         min    max
kelas              
Hawar  10:08  16:26
Karat  08:54  16:45
Sehat  16:27  16:40


In [11]:
# ---- 11b. session hold-out untuk kelas sehat
Pall = np.concatenate([Ptr, Pte]); yall = np.concatenate([ytr, yte])
hid = classes.index("Sehat")
tgl = meta[meta.kelas == "Sehat"].tanggal.value_counts()
print("tanggal kelas sehat:", dict(tgl))
if len(tgl) > 1:
    minor = tgl.index[-1]
    held = ((meta.kelas == "Sehat") & (meta.tanggal == minor)).values
    res = []
    for s in range(5):
        h = fit_head(Pall[~held], yall[~held], seed=SEED + s)
        pr = h.predict(Pall[held], verbose=0).argmax(1)
        res.append((pr == hid).mean())
    print(f"sesi {minor}: {held.sum()} citra sehat tidak pernah dilihat; diprediksi sehat: {100*np.mean(res):.1f} +- {100*np.std(res, ddof=1):.1f}% (5 benih)")
    json.dump(dict(sesi_ditahan=minor, n=int(held.sum()), recall_sehat=float(np.mean(res)), sd=float(np.std(res, ddof=1))),
              open(os.path.join(OUT_DIR, "session_holdout.json"), "w"))
else:
    print("Hanya satu tanggal pada kelas sehat; session hold-out tidak dapat dilakukan.")


tanggal kelas sehat: {'2023-03-15': np.int64(101)}
Hanya satu tanggal pada kelas sehat; session hold-out tidak dapat dilakukan.


In [12]:
# ---- 11c. kontrol warna: grayscale dan standardisasi per citra
def ctrl_img(X, mode):
    X = X.astype("float32")
    if mode == "gray":
        g = X @ np.array([.299, .587, .114], dtype="float32")
        return np.repeat(g[..., None], 3, -1)
    m = X.mean((1, 2), keepdims=True); s = X.std((1, 2), keepdims=True) + 1e-6      # zscore per citra dan per kanal
    return np.clip((X - m) / s * 50 + 128, 0, 255)
def feats_ctrl(X, mode, bs=64):
    out = []
    for i in range(0, len(X), bs):
        x = keras.applications.resnet50.preprocess_input(ctrl_img(X[i:i + bs], mode))
        out.append(pool(f2.predict(x, batch_size=16, verbose=0)))
    return np.concatenate(out)

def cv_F(Ptr_, Pte_):
    va, te = [], []
    for fold, (itr, iva) in enumerate(skf.split(Xtr, ytr), 1):
        h = fit_head(Ptr_[itr], ytr[itr], Xv=Ptr_[iva], yv=ytr[iva], seed=SEED + fold)
        va.append(accuracy_score(ytr[iva], h.predict(Ptr_[iva], verbose=0).argmax(1)))
        te.append(accuracy_score(yte, h.predict(Pte_, verbose=0).argmax(1)))
    return np.array(va), np.array(te)

rows_c = []
for name, (A, B) in {"asli (RGB)": (Ptr, Pte),
                     "grayscale": (feats_ctrl(Xtr, "gray"), feats_ctrl(Xte, "gray")),
                     "standardisasi per citra": (feats_ctrl(Xtr, "zscore"), feats_ctrl(Xte, "zscore"))}.items():
    va, te = cv_F(A, B)
    rows_c.append({"Masukan": name, "Val acc (%)": ms(pd.Series(va)), "Test acc (%)": ms(pd.Series(te))})
    log("kontrol", name, rows_c[-1])
ctrl = pd.DataFrame(rows_c); ctrl.to_csv(os.path.join(OUT_DIR, "kontrol_akuisisi.csv"), index=False)
print(ctrl.to_string(index=False))


[  2356s] kontrol asli (RGB) {'Masukan': 'asli (RGB)', 'Val acc (%)': '94.3 ± 1.7', 'Test acc (%)': '93.4 ± 1.3'}
[  2409s] kontrol grayscale {'Masukan': 'grayscale', 'Val acc (%)': '89.8 ± 1.4', 'Test acc (%)': '88.1 ± 2.4'}
[  2505s] kontrol standardisasi per citra {'Masukan': 'standardisasi per citra', 'Val acc (%)': '93.1 ± 4.0', 'Test acc (%)': '93.4 ± 2.0'}
                Masukan Val acc (%) Test acc (%)
             asli (RGB)  94.3 ± 1.7   93.4 ± 1.3
              grayscale  89.8 ± 1.4   88.1 ± 2.4
standardisasi per citra  93.1 ± 4.0   93.4 ± 2.0


## 12. Perbandingan dengan arsitektur ringan (catatan reviewer 3)

ResNet50, MobileNetV2, dan EfficientNetB0 dievaluasi dengan protokol yang sama: pembagian 80:20 seed 69, validasi silang 5-fold, kepala klasifikasi yang sama pada fitur beku (konfigurasi F), tanpa dan dengan augmentasi dari citra latih fold. Tabel juga memuat jumlah parameter dan waktu inferensi per citra pada perangkat Colab yang dipakai.

In [13]:
# ---- 12. ResNet50 vs MobileNetV2 vs EfficientNetB0 (fitur beku + kepala yang sama)
ARCHS = {"ResNet50": (keras.applications.ResNet50, keras.applications.resnet50.preprocess_input),
         "MobileNetV2": (keras.applications.MobileNetV2, keras.applications.mobilenet_v2.preprocess_input),
         "EfficientNetB0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input)}
def gap_feats(model, prep, X, bs=32):
    return np.concatenate([model.predict(prep(X[i:i + bs].astype("float32")), batch_size=bs, verbose=0)
                           for i in range(0, len(X), bs)])
def cv_gen(Ptr_, Paug_, Pte_, aug):
    rr = []
    for fold, (itr, iva) in enumerate(skf.split(Xtr, ytr), 1):
        if aug:
            sel = np.isin(src_idx, itr)
            Xp_ = np.concatenate([Ptr_[itr], Paug_[sel]]); yy = np.concatenate([ytr[itr], ytr[src_idx[sel]]])
        else:
            Xp_, yy = Ptr_[itr], ytr[itr]
        h = fit_head(Xp_, yy, Xv=Ptr_[iva], yv=ytr[iva], seed=SEED + fold)
        rr.append(dict(val_acc=accuracy_score(ytr[iva], h.predict(Ptr_[iva], verbose=0).argmax(1)),
                       **metrics(yte, h.predict(Pte_, verbose=0))))
    return pd.DataFrame(rr)

MODELS, FEATS, rows_a = {}, {}, []
for name, (ctor, prep) in ARCHS.items():
    m = ctor(include_top=False, weights=WEIGHTS, input_shape=(IMG, IMG, 3), pooling="avg")
    MODELS[name] = (m, prep)
    Ftr_, Faug_, Fte_ = gap_feats(m, prep, Xtr), gap_feats(m, prep, Xaug), gap_feats(m, prep, Xte)
    FEATS[name] = (Ftr_, Faug_, Fte_)
    t0 = time.time(); m.predict(prep(Xte[:32].astype("float32")), batch_size=1, verbose=0); ms_img = 1000 * (time.time() - t0) / 32
    for aug in (False, True):
        g = cv_gen(Ftr_, Faug_, Fte_, aug)
        rows_a.append({"Model": name, "Augmentasi": "ya" if aug else "tidak", "Parameter (juta)": round(m.count_params() / 1e6, 1),
                       "ms/citra": round(ms_img, 1), "Val acc (%)": ms(g.val_acc), "Test acc (%)": ms(g.acc),
                       "Test F1 (w)": ms(g.f1_w, False), "Test AUC": ms(g.auc, False)})
        log(name, rows_a[-1])
arch = pd.DataFrame(rows_a); arch.to_csv(os.path.join(OUT_DIR, "perbandingan_arsitektur.csv"), index=False)
print(arch.to_string(index=False))


[  2824s] ResNet50 {'Model': 'ResNet50', 'Augmentasi': 'tidak', 'Parameter (juta)': 23.6, 'ms/citra': 162.3, 'Val acc (%)': '94.3 ± 1.7', 'Test acc (%)': '93.4 ± 1.3', 'Test F1 (w)': '0.934 ± 0.013', 'Test AUC': '0.983 ± 0.004'}
[  2979s] ResNet50 {'Model': 'ResNet50', 'Augmentasi': 'ya', 'Parameter (juta)': 23.6, 'ms/citra': 162.3, 'Val acc (%)': '94.7 ± 3.3', 'Test acc (%)': '93.4 ± 0.7', 'Test F1 (w)': '0.934 ± 0.007', 'Test AUC': '0.987 ± 0.006'}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[  3099s] MobileNetV2 {'Model': 'MobileNetV2', 'Augmentasi': 'tidak', 'Parameter (juta)': 2.3, 'ms/citra': 38.7, 'Val acc (%)': '87.4 ± 2.7', 'Test acc (%)': '87.5 ± 2.5', 'Test F1 (w)': '0.874 ± 0.026', 'Test AUC': '0.970 ± 0.005'}
[  3235s] MobileNetV2 {'Model': 'MobileNetV2', 'Augmentasi': 'ya', 'Parameter (juta)': 2.3, 'ms/citra': 38.7, 'Val acc (%)': '89.4 ± 2.2', 'Test acc (%)': '88.4 ± 3.0', 'Test F1 (w)': '0.884 ± 0.030', 'Test AUC': '0.973 ± 0.011'}
16705208/16705208 ━━━━━━━━━━━━━━━

## 13. Dataset publik: PlantVillage jagung (catatan reviewer 3)

Subset jagung PlantVillage (Northern Leaf Blight, Common Rust, Healthy) dipetakan ke `Hawar`, `Karat`, `Sehat`; kelas Gray Leaf Spot tidak dipakai. Asumsi pemetaan `Hawar` = Northern Leaf Blight harus dikonfirmasi oleh penulis dari data lapangan.

Tiga evaluasi: (i) validasi silang di dalam PlantVillage (pembanding), (ii) latih di PlantVillage, uji di data lokal (310 citra), (iii) latih di data lokal (246 citra latih), uji di PlantVillage. Skenario (ii) dan (iii) mengukur generalisasi lintas domain. Perangkat Colab butuh unduhan sekitar 0,8 GB untuk `plant_village`; jumlah citra per kelas diatur oleh `PV_N`.

In [18]:
!pip -q install tensorflow-datasets

In [22]:
# ---- 13. PlantVillage jagung dan uji lintas dataset
!pip -q install importlib_resources
import tensorflow_datasets as tfds
PV_N = int(os.environ.get("PV_N", 150))
ds, info = tfds.load("plant_village", split="train", with_info=True, as_supervised=True)
names = info.features["label"].names
def find(sub):
    m = [n for n in names if "corn" in n.lower() and sub in n.lower()]
    assert len(m) == 1, (sub, m); return names.index(m[0])
ids = {"Hawar": find("northern"), "Karat": find("rust"), "Sehat": find("healthy")}
inv = {v: k for k, v in ids.items()}
ds_f = ds.filter(lambda im, l: tf.reduce_any(tf.equal(l, tf.constant(list(ids.values()), tf.int64))))
Xp, yp, cnt = [], [], {k: 0 for k in ids}
for im, l in tfds.as_numpy(ds_f.shuffle(5000, seed=SEED)):
    k = inv[int(l)]
    if cnt[k] >= PV_N: continue
    Xp.append(cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)); yp.append(classes.index(k)); cnt[k] += 1
    if min(cnt.values()) >= PV_N: break
Xp, yp = np.stack(Xp), np.array(yp)
log("PlantVillage jagung:", Xp.shape, np.bincount(yp), "| kelas lokal:", classes)

rows_x = []
for name, (m, prep) in MODELS.items():
    Ftr_, _, Fte_ = FEATS[name]; Floc = np.concatenate([Ftr_, Fte_]); yloc = np.concatenate([ytr, yte])
    Fpv = gap_feats(m, prep, Xp)
    # (i) dalam domain PlantVillage: 5-fold
    acc_in = []
    for fold, (a, b) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(Fpv, yp), 1):
        h = fit_head(Fpv[a], yp[a], seed=SEED + fold); acc_in.append(accuracy_score(yp[b], pred_p(h, Fpv[b]).argmax(1)))
    # (ii) PV -> lokal dan (iii) lokal(latih) -> PV, 3 benih
    a2, a3 = [], []
    for s in range(3):
        h = fit_head(Fpv, yp, seed=SEED + s); a2.append(accuracy_score(yloc, pred_p(h, Floc).argmax(1)))
        h = fit_head(Ftr_, ytr, seed=SEED + s); a3.append(accuracy_score(yp, pred_p(h, Fpv).argmax(1)))
    rows_x.append({"Model": name, "PV dalam domain (%)": ms(pd.Series(acc_in)), "PV -> lokal (%)": ms(pd.Series(a2)),
                   "Lokal -> PV (%)": ms(pd.Series(a3))})
    log(name, rows_x[-1])
cross = pd.DataFrame(rows_x); cross.to_csv(os.path.join(OUT_DIR, "lintas_dataset.csv"), index=False)
print(cross.to_string(index=False))

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.237R2K_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
[  5939s] PlantVillage jagung: (450, 224, 224, 3) [150 150 150] | kelas lokal: ['Hawar', 'Karat', 'Sehat']
[  6231s] ResNet50 {'Model': 'ResNet50', 'PV dalam domain (%)': '100.0 ± 0.0', 'PV -> lokal (%)': '48.6 ± 3.6', 'Lokal -> PV (%)': '69.6 ± 4.7'}
[  6429s] MobileNetV2 {'Model': 'MobileNetV2', 'PV dalam domain (%)': '99.1 ± 0.9', 'PV -> lokal (%)': '38.1 ± 1.2', 'Lokal -> PV (%)': '66.3 ± 2.8'}
[  6673s] EfficientNetB0 {'Model': 'EfficientNetB0', 'PV dalam domain (%)': '99.6 ± 1.0', 'PV -> lokal (%)': '34.5 ± 0.0', 'Lokal -> PV (%)': '82.6 ± 4.4'}
         Model PV dalam domain (%) PV -> lokal (%) Lokal -> PV (%)
      ResNet50         100.0 ± 0.0      48.6 ± 3.6      69.6 ± 4.7
   MobileNetV2          99.1 ± 0.9      38.1 ± 1.2      66.3 ± 2.8
EfficientNetB0          99.6 ± 1.0      34.5 ± 0.0      82.6 ± 4.4


## 14. Unduh hasil

In [ ]:
!zip -qr hasil_resnet50.zip hasil_resnet50 -x 'hasil_resnet50/split/*' 'hasil_resnet50/cache.npz'
from google.colab import files
files.download('hasil_resnet50.zip')